In [ ]:
import numpy as np

file_path = "F:\\research project\\# study 1\\eeg_data\\set\\Y group\\source_pac_stats\\PAC_MI_SOURCE_Y_BL_go.npy"
pac_data = np.load(file_path)
pac_data

array([[[[2.97064782e-04, 2.92688142e-04, 2.84926287e-04, ...,
          1.27594813e-04, 1.45962540e-04, 1.70725423e-04],
         [2.91073284e-04, 2.98423631e-04, 2.89674750e-04, ...,
          1.03539545e-04, 1.16161596e-04, 1.52568011e-04],
         [3.49872666e-04, 3.06424302e-04, 2.74908640e-04, ...,
          1.01776918e-04, 1.16246851e-04, 1.49539291e-04],
         ...,
         [1.45015625e-04, 8.65785085e-05, 5.26893209e-05, ...,
          6.79390990e-05, 1.13159260e-04, 1.52594726e-04],
         [1.39620060e-04, 8.91586309e-05, 5.38408943e-05, ...,
          5.77872621e-05, 8.83499511e-05, 1.11354055e-04],
         [1.12368489e-04, 7.62790996e-05, 5.46462271e-05, ...,
          3.88751499e-05, 5.98987945e-05, 7.71683455e-05]],

        [[1.16877724e-04, 1.58645925e-04, 2.52955402e-04, ...,
          2.38033754e-04, 2.40674834e-04, 2.14956871e-04],
         [9.89782751e-05, 2.00916366e-04, 2.97630274e-04, ...,
          8.73589946e-05, 8.81972021e-05, 7.80387448e-05],
        

SURROGATES TEST (1 participant, 1 condition)

In [15]:
from pathlib import Path
import numpy as np
import mne
from pactools import Comodulogram

# ============================================================
# 1. SETTINGS
# ============================================================

sub_name = "s1_pac_sub01"
task = "_BL"
task_stage = "_go"
# block = "_adaptation"

data_dir = Path(
    r"F:\\research project\\# study 1\\eeg_data\\set"
)

# UPDATE: Path to the parent directory containing fsaverage_bem
subjects_dir = Path(
    r"F:\\research project\\# study 1\\mri_data\\fs_output\\freesurfer\\sub_dir\\Y"
)

# UPDATE: Select one name from your original roi_label_names
ROI_NAME = "G_precentral-lh"

n_surrogates = 200  # Pilot only

theta_range = np.linspace(4, 8, 20)
gamma_range = np.linspace(30, 80, 20)

# ============================================================
# 2. LOAD ROI LABEL
# ============================================================

labels = mne.read_labels_from_annot(
    subject="fsaverage_bem",
    parc="aparc.a2009s",
    subjects_dir=str(subjects_dir)
)

roi_matches = [lab for lab in labels if lab.name == ROI_NAME]

if len(roi_matches) != 1:
    raise ValueError(
        f"ROI {ROI_NAME!r} not found. "
        f"Available names: {[lab.name for lab in labels]}"
    )

roi_label = roi_matches[0]

print("Selected ROI:", roi_label.name)

# ============================================================
# 3. LOAD MORPHED SOURCE ESTIMATES
# ============================================================

stc_dir = (
    data_dir / "Y" / sub_name
    / "preproc" / "analysis" / "source"
    / "morphed_stcs" / task / task_stage
)

stc_files = sorted(stc_dir.glob("*_morphed-lh.stc"))

if not stc_files:
    raise FileNotFoundError(f"No STC files found in {stc_dir}")

roi_epochs = []
sfreq = None

for file in stc_files:

    # MNE automatically reads both hemispheres
    stc = mne.read_source_estimate(str(file))

    # Match the original post-event interval
    stc = stc.copy().crop(tmin=0.0, tmax=0.495)

    # Extract source signals for vertices belonging to the ROI
    stc_roi = stc.in_label(roi_label)

    if sfreq is None:
        sfreq = stc.sfreq

    roi_epochs.append(stc_roi.data)

roi_epochs = np.stack(roi_epochs, axis=0)

# Shape: epochs x ROI vertices x time
print("ROI data shape:", roi_epochs.shape)
print("Sampling frequency:", sfreq)

# # ============================================================
# # 4. CALCULATE ORIGINAL AND SURROGATE PAC
# # ============================================================

# n_epochs, n_vertices, n_times = roi_epochs.shape

# pac_original = np.empty(n_vertices)
# pac_surrogates = np.empty((n_vertices, n_surrogates))

# for vertex in range(n_vertices):

#     print(f"Processing vertex {vertex + 1}/{n_vertices}")

#     # Same epoch concatenation as in the original analysis
#     data_flat = roi_epochs[:, vertex, :].reshape(1, -1)

#     estimator = Comodulogram(
#         fs=sfreq,
#         low_fq_range=theta_range,
#         high_fq_range=gamma_range,
#         method="tort",
#         n_surrogates=n_surrogates,
#         minimum_shift=1.0,
#         random_state=42 + vertex,
#         progress_bar=False
#     )

#     estimator.fit(data_flat, data_flat)

#     # Average over the 20 x 20 frequency pairs
#     pac_original[vertex] = np.mean(estimator.comod_)

#     # Shape: n_surrogates x 20 x 20
#     pac_surrogates[vertex] = np.mean(
#         estimator.surrogates_,
#         axis=(1, 2)
#     )

# # ============================================================
# # 5. AVERAGE ACROSS ROI VERTICES
# # ============================================================

# roi_pac_original = np.mean(pac_original)

# # One ROI-level value for each surrogate iteration
# roi_pac_surrogates = np.mean(
#     pac_surrogates,
#     axis=0
# )

# print("\nRESULTS")
# print("Original ROI PAC:", roi_pac_original)
# print("Mean surrogate PAC:", roi_pac_surrogates.mean())
# print("Surrogate SD:", roi_pac_surrogates.std())

# # Exploratory z-score (20 surrogates are insufficient for inference)
# roi_pac_z = (
#     roi_pac_original - roi_pac_surrogates.mean()
# ) / roi_pac_surrogates.std()

# print("Exploratory surrogate-normalized z:", roi_pac_z)

# # ============================================================
# # 6. SAVE PILOT RESULTS
# # ============================================================

# output_dir = stc_dir / "surrogate_pilot"
# output_dir.mkdir(exist_ok=True)

# output_file = output_dir / f"{sub_name}_{ROI_NAME}_pilot.npz"

# np.savez(
#     output_file,
#     original=roi_pac_original,
#     surrogates=roi_pac_surrogates,
#     roi=ROI_NAME,
#     n_epochs=n_epochs,
#     n_vertices=n_vertices,
#     n_surrogates=n_surrogates
# )

# print("\nSaved:", output_file)


Reading labels from parcellation...
   read 75 labels from F:\research project\# study 1\mri_data\fs_output\freesurfer\sub_dir\Y\fsaverage_bem\label\lh.aparc.a2009s.annot
   read 75 labels from F:\research project\# study 1\mri_data\fs_output\freesurfer\sub_dir\Y\fsaverage_bem\label\rh.aparc.a2009s.annot
Selected ROI: G_precentral-lh
ROI data shape: (84, 60, 249)
Sampling frequency: 500.0


In [16]:
# ============================================================
# 4. CALCULATE ORIGINAL AND SURROGATE PAC
# ============================================================

n_epochs, n_vertices, n_times = roi_epochs.shape

pac_original = np.empty(n_vertices)
pac_surrogates = np.empty((n_vertices, n_surrogates))

for vertex in range(n_vertices):

    print(f"Processing vertex {vertex + 1}/{n_vertices}")

    # Same epoch concatenation as in the original analysis
    data_flat = roi_epochs[:, vertex, :].reshape(1, -1)

    estimator = Comodulogram(
        fs=sfreq,
        low_fq_range=theta_range,
        high_fq_range=gamma_range,
        method="tort",
        n_surrogates=n_surrogates,
        minimum_shift=1.0,
        random_state=42 + vertex,
        progress_bar=False
    )

    estimator.fit(data_flat, data_flat)

    # Average over the 20 x 20 frequency pairs
    pac_original[vertex] = np.mean(estimator.comod_)

    # Shape: n_surrogates x 20 x 20
    pac_surrogates[vertex] = np.mean(
        estimator.surrogates_,
        axis=(1, 2)
    )

Processing vertex 1/60
Processing vertex 2/60
Processing vertex 3/60
Processing vertex 4/60
Processing vertex 5/60
Processing vertex 6/60
Processing vertex 7/60
Processing vertex 8/60
Processing vertex 9/60
Processing vertex 10/60
Processing vertex 11/60
Processing vertex 12/60
Processing vertex 13/60
Processing vertex 14/60
Processing vertex 15/60
Processing vertex 16/60
Processing vertex 17/60
Processing vertex 18/60
Processing vertex 19/60
Processing vertex 20/60
Processing vertex 21/60
Processing vertex 22/60
Processing vertex 23/60
Processing vertex 24/60
Processing vertex 25/60
Processing vertex 26/60
Processing vertex 27/60
Processing vertex 28/60
Processing vertex 29/60
Processing vertex 30/60
Processing vertex 31/60
Processing vertex 32/60
Processing vertex 33/60
Processing vertex 34/60
Processing vertex 35/60
Processing vertex 36/60
Processing vertex 37/60
Processing vertex 38/60
Processing vertex 39/60
Processing vertex 40/60
Processing vertex 41/60
Processing vertex 42/60
P

In [17]:
print("PAC original shape:", pac_original.shape)
print("PAC surrogates shape:", pac_surrogates.shape)

PAC original shape: (60,)
PAC surrogates shape: (60, 200)


In [18]:
# ============================================================
# 5. AVERAGE ACROSS ROI VERTICES
# ============================================================

roi_pac_original = np.mean(pac_original)

# One ROI-level value for each surrogate iteration
roi_pac_surrogates = np.mean(
    pac_surrogates,
    axis=0
)

print("\nRESULTS")
print("Original ROI PAC:", roi_pac_original)
print("Mean surrogate PAC:", roi_pac_surrogates.mean())
print("Surrogate SD:", roi_pac_surrogates.std())

# Exploratory z-score (20 surrogates are insufficient for inference)
roi_pac_z = (
    roi_pac_original - roi_pac_surrogates.mean()
) / roi_pac_surrogates.std()

print("Exploratory surrogate-normalized z:", roi_pac_z)


RESULTS
Original ROI PAC: 0.00016285398689487438
Mean surrogate PAC: 0.00016076313437257607
Surrogate SD: 4.198019662933089e-06
Exploratory surrogate-normalized z: 0.49805686732716825


Check whether the previous and new PAC outputs are comparable

In [ ]:
# Load previously calculated source-level PAC
pac_file = (
    data_dir / "Y" / sub_name
    / "preproc" / "analysis" / "source" / "PAC"
    / f"PAC_MI_SOURCE_{sub_name[-5:]}{task}{task_stage}.npy"
)

pac_saved = np.load(pac_file)

print("Previously saved PAC shape:", pac_saved.shape)
print("New original ROI PAC:", roi_pac_original)

Previously saved PAC shape: (20, 5124, 20)
New original ROI PAC: 0.00016285398689487438


In [ ]:
# Find indices of vertices belonging to our ROI
roi_indices = np.where(
    np.isin(stc.vertices[0], roi_label.vertices)
)[0]

print("Number of ROI vertices:", len(roi_indices))

# Extract PAC for these vertices
pac_saved_roi = pac_saved[:, roi_indices, :]

# Average across frequencies and vertices
roi_pac_saved = np.mean(pac_saved_roi)

print("\nPreviously saved ROI PAC:", roi_pac_saved)
print("Newly calculated ROI PAC:", roi_pac_original)

print(
    "Difference:",
    roi_pac_original - roi_pac_saved
)

Number of ROI vertices: 60

Previously saved ROI PAC: 0.00016285398689487438
Newly calculated ROI PAC: 0.00016285398689487438
Difference: 0.0


SELECTING APPROPRIATE SURROGATE GENERATION PROCEDURE

In [19]:
# ============================================================
# TRIAL-SHUFFLED SURROGATE PAC
# ============================================================

rng = np.random.default_rng(42)

n_epochs, n_vertices, n_times = roi_epochs.shape
n_surrogates = 200

pac_trial_shuffled = np.empty((n_vertices, n_surrogates))

# Generate the same trial shifts for all ROI vertices
# Every shift is nonzero, so no trial remains paired with itself
trial_shifts = rng.integers(
    1, n_epochs, size=n_surrogates
)

for vertex in range(n_vertices):

    print(f"Processing vertex {vertex + 1}/{n_vertices}")

    # Original signal for theta phase
    data_low = roi_epochs[:, vertex, :].reshape(1, -1)

    for s, shift in enumerate(trial_shifts):

        # Reorder trials for gamma amplitude
        shuffled_epochs = np.roll(
            roi_epochs[:, vertex, :],
            shift=shift,
            axis=0
        )

        data_high = shuffled_epochs.reshape(1, -1)

        estimator = Comodulogram(
            fs=sfreq,
            low_fq_range=theta_range,
            high_fq_range=gamma_range,
            method="tort",
            n_surrogates=0,
            progress_bar=False
        )

        estimator.fit(data_low, data_high)

        pac_trial_shuffled[vertex, s] = np.mean(
            estimator.comod_
        )

# Average across ROI vertices
roi_pac_trial_shuffled = np.mean(
    pac_trial_shuffled,
    axis=0
)

# ============================================================
# COMPARE METHODS
# ============================================================

print("\nORIGINAL PAC")
print("Original ROI PAC:", roi_pac_original)

print("\nTIME-SHIFTED SURROGATES")
print("Mean:", roi_pac_surrogates.mean())
print("SD:", roi_pac_surrogates.std())

print("\nTRIAL-SHUFFLED SURROGATES")
print("Mean:", roi_pac_trial_shuffled.mean())
print("SD:", roi_pac_trial_shuffled.std())

print("\nSHAPES")
print("Time-shifted:", roi_pac_surrogates.shape)
print("Trial-shuffled:", roi_pac_trial_shuffled.shape)

Processing vertex 1/60
Processing vertex 2/60
Processing vertex 3/60
Processing vertex 4/60
Processing vertex 5/60
Processing vertex 6/60
Processing vertex 7/60
Processing vertex 8/60
Processing vertex 9/60
Processing vertex 10/60
Processing vertex 11/60
Processing vertex 12/60
Processing vertex 13/60
Processing vertex 14/60
Processing vertex 15/60
Processing vertex 16/60
Processing vertex 17/60
Processing vertex 18/60
Processing vertex 19/60
Processing vertex 20/60
Processing vertex 21/60
Processing vertex 22/60
Processing vertex 23/60
Processing vertex 24/60
Processing vertex 25/60
Processing vertex 26/60
Processing vertex 27/60
Processing vertex 28/60
Processing vertex 29/60
Processing vertex 30/60
Processing vertex 31/60
Processing vertex 32/60
Processing vertex 33/60
Processing vertex 34/60
Processing vertex 35/60
Processing vertex 36/60
Processing vertex 37/60
Processing vertex 38/60
Processing vertex 39/60
Processing vertex 40/60
Processing vertex 41/60
Processing vertex 42/60
P